<!-- # NIST SP 800-22 による乱数の統計的検定 -->

# Statistical testing of random numbers with NIST SP 800-22

<!-- ## このノートブックで行うこと -->

## What this notebook does

<!-- ビット列が乱数らしいかを、NIST SP 800-22 Rev. 1a の 15 種類の統計的検定で調べます。検定には NIST が配布している公式ツール **STS（Statistical Test Suite）2.1.2** を使います。 -->

We examine whether a bit string looks random using the 15 statistical tests of NIST SP 800-22 Rev. 1a. For the tests we use the official tool distributed by NIST, **STS (Statistical Test Suite) 2.1.2**.

> L. E. Bassham III *et al.*, "A Statistical Test Suite for Random and Pseudorandom Number Generators for Cryptographic Applications," NIST Special Publication 800-22 Revision 1a (2010).

<!--
1. 公式ツール STS の使い方を確認する（1.）。STS のダウンロードとビルドは `README.md` の手順で行う
2. 検定する系列（$10^6$ ビット × 55 本）を STS の入力ファイルに書く。Qiskit で C1 回路の QRNG をシミュレーションするか、保存したビット列（抽出器の出力など）を使う（3.）
3. STS で 15 種類の検定を行い、報告の 188 行を 2 つの基準で読む（4.）
4. 結果を CSV に保存する（5.）
-->

1. Check how to use the official tool STS (1.). Download and build STS by following `README.md`
2. Write the sequences to be tested ($10^6$ bits × 55) to the STS input file: either simulate the C1-circuit QRNG with Qiskit, or use a saved bit string (such as the output of an extractor) (3.)
3. Run the 15 tests with STS and read the 188 rows of the report against the two criteria (4.)
4. Save the results to CSV (5.)

---

<!-- ## 手法 -->

## Methods

<!-- ### A. 仮説検定と P 値 -->

### A. Hypothesis testing and P-values

<!-- 各検定は、帰無仮説 $H_0$「ビット列は乱数である（各ビットが独立に確率 1/2 で 0 と 1 をとる）」の検定です。ビット列から検定統計量を計算し、$H_0$ のもとでそれと同じかそれ以上に乱数らしくない値が出る確率（P 値）を求めます。有意水準は $\alpha = 0.01$ で、$\text{P 値} \ge \alpha$ なら「乱数らしい」と判定します（SP 800-22 の 1.1.5 節）。 -->

Each test is a test of the null hypothesis $H_0$: "the bit string is random (each bit independently takes 0 and 1 with probability 1/2)". A test statistic is computed from the bit string, and the probability under $H_0$ of a value as non-random as that or more (the P-value) is obtained. The significance level is $\alpha = 0.01$, and the sequence is judged "random-looking" if $\text{P-value} \ge \alpha$ (Section 1.1.5 of SP 800-22).

<!-- ### B. 15 種類の検定（SP 800-22 の 2 章） -->

### B. The 15 tests (Chapter 2 of SP 800-22)

<!-- 長さ $n$ のビット列を $\varepsilon_1, \ldots, \varepsilon_n$、$X_i = 2\varepsilon_i - 1$（$\pm 1$）とします。$\mathrm{erfc}$ は相補誤差関数、$\mathrm{igamc}(a, x)$ は正規化された上側不完全ガンマ関数（自由度 $2a$ のカイ二乗分布の上側確率が $\mathrm{igamc}(a, \chi^2/2)$）、$\Phi$ は標準正規分布の累積分布関数です。パラメータは STS の既定値と、$n = 10^6$ での値です。 -->

Let the bit string of length $n$ be $\varepsilon_1, \ldots, \varepsilon_n$, and $X_i = 2\varepsilon_i - 1$ ($\pm 1$). $\mathrm{erfc}$ is the complementary error function, $\mathrm{igamc}(a, x)$ is the regularized upper incomplete gamma function (the upper-tail probability of the chi-squared distribution with $2a$ degrees of freedom is $\mathrm{igamc}(a, \chi^2/2)$), and $\Phi$ is the cumulative distribution function of the standard normal distribution. The parameters are the STS defaults and the values for $n = 10^6$.

<!--
| 節 | 検定（STS の名前） | 検定統計量と P 値 | 簡単な説明 | 行数 |
|---|---|---|---|---|
| 2.1 | Frequency (Monobit) (`Frequency`) | $s = \lvert\sum_i X_i\rvert/\sqrt{n}$、$P = \mathrm{erfc}(s/\sqrt{2})$ | 0 と 1 の個数の差 | 1 |
| 2.2 | Frequency within a Block (`BlockFrequency`) | $\chi^2 = 4M\sum_{j=1}^{N}(\pi_j - \tfrac{1}{2})^2$、$P = \mathrm{igamc}(N/2, \chi^2/2)$ | $M = 128$ ビットのブロック $j$ の 1 の割合 $\pi_j$（$N = \lfloor n/M \rfloor$） | 1 |
| 2.3 | Runs (`Runs`) | $P = \mathrm{erfc}\left(\dfrac{\lvert V - 2n\pi(1-\pi)\rvert}{2\sqrt{2n}\,\pi(1-\pi)}\right)$ | 同じ値の連続（run）の数 $V$。$\pi$ は 1 の割合で、$\lvert\pi - \tfrac{1}{2}\rvert > 2/\sqrt{n}$ なら $P = 0$ | 1 |
| 2.4 | Longest Run of Ones in a Block (`LongestRun`) | $\chi^2 = \sum_{i=0}^{6}(\nu_i - N\pi_i)^2/(N\pi_i)$、$P = \mathrm{igamc}(3, \chi^2/2)$ | $M = 10^4$ ビットのブロックの 1 の最長の run を 7 つの区分（10 以下, 11, …, 15, 16 以上）に数えた $\nu_i$ | 1 |
| 2.5 | Binary Matrix Rank (`Rank`) | $\chi^2 = \sum_{r}(F_r - Np_r)^2/(Np_r)$、$P = e^{-\chi^2/2}$ | $32 \times 32$ の行列の GF(2) 上の階数が 32, 31, 30 以下の個数 $F_r$（$p_{32} \approx 0.2888$, $p_{31} \approx 0.5776$, $p_{30} \approx 0.1336$） | 1 |
| 2.6 | Discrete Fourier Transform (`FFT`) | $d = \dfrac{N_1 - 0.95n/2}{\sqrt{n \cdot 0.95 \cdot 0.05/4}}$、$P = \mathrm{erfc}(\lvert d\rvert/\sqrt{2})$ | $X$ の離散フーリエ変換の大きさ（前半 $n/2$ 個）のうち $T = \sqrt{n\ln 20}$ 未満の個数 $N_1$。周期的な構造を見る | 1 |
| 2.7 | Non-overlapping Template Matching (`NonOverlappingTemplate`) | $\chi^2 = \sum_{j=1}^{8}(W_j - \mu)^2/\sigma^2$、$P = \mathrm{igamc}(4, \chi^2/2)$ | 8 個のブロック（$M = \lfloor n/8 \rfloor$）で、周期をもたない $m = 9$ ビットのパターンが重ならずに現れる回数 $W_j$。$\mu = (M - m + 1)/2^m$、$\sigma^2 = M(2^{-m} - (2m-1)2^{-2m})$ | 148 |
| 2.8 | Overlapping Template Matching (`OverlappingTemplate`) | $\chi^2 = \sum_{i=0}^{5}(\nu_i - N\pi_i)^2/(N\pi_i)$、$P = \mathrm{igamc}(5/2, \chi^2/2)$ | $M = 1032$ ビットのブロックで、1 が 9 個続くパターンが重なりを許して現れる回数（0, 1, …, 4, 5 以上）のブロック数 $\nu_i$ | 1 |
| 2.9 | Maurer's "Universal Statistical" (`Universal`) | $P = \mathrm{erfc}\left(\dfrac{\lvert f_n - \mathrm{E}[L]\rvert}{\sqrt{2}\,\sigma}\right)$ | $L = 7$ ビットのブロックが前回出てから何ブロック後に出たかの $\log_2$ の平均 $f_n$（最初の $10 \cdot 2^L$ ブロックは初期化に使う）。圧縮のしやすさを見る | 1 |
| 2.10 | Linear Complexity (`LinearComplexity`) | $\chi^2 = \sum_{i=0}^{6}(\nu_i - N\pi_i)^2/(N\pi_i)$、$P = \mathrm{igamc}(3, \chi^2/2)$ | $M = 500$ ビットのブロックの線形複雑度 $L_j$（Berlekamp–Massey 法）から $T_j = (-1)^M(L_j - \mu) + 2/9$ を 7 つの区分に数えた $\nu_i$ | 1 |
| 2.11 | Serial (`Serial`) | $P_1 = \mathrm{igamc}(2^{m-2}, \nabla\psi^2_m/2)$、$P_2 = \mathrm{igamc}(2^{m-3}, \nabla^2\psi^2_m/2)$ | 重なりを許す $m = 16$ ビットのパターンの頻度 $\nu$ から $\psi^2_m = \frac{2^m}{n}\sum \nu^2 - n$、$\nabla\psi^2_m = \psi^2_m - \psi^2_{m-1}$、$\nabla^2\psi^2_m = \psi^2_m - 2\psi^2_{m-1} + \psi^2_{m-2}$ | 2 |
| 2.12 | Approximate Entropy (`ApproximateEntropy`) | $\chi^2 = 2n(\ln 2 - \mathrm{ApEn})$、$P = \mathrm{igamc}(2^{m-1}, \chi^2/2)$ | $\mathrm{ApEn} = \varphi^{(m)} - \varphi^{(m+1)}$、$\varphi^{(m)} = \sum_i \frac{C_i}{n}\ln\frac{C_i}{n}$（$C_i$ は重なりを許す $m = 10$ ビットのパターン $i$ の回数） | 1 |
| 2.13 | Cumulative Sums (`CumulativeSums`) | $P = 1 - \sum_k\bigl[\Phi(\tfrac{(4k+1)z}{\sqrt{n}}) - \Phi(\tfrac{(4k-1)z}{\sqrt{n}})\bigr] + \sum_k\bigl[\Phi(\tfrac{(4k+3)z}{\sqrt{n}}) - \Phi(\tfrac{(4k+1)z}{\sqrt{n}})\bigr]$ | ランダムウォーク $S_k = \sum_{i \le k} X_i$ の最大の振れ幅 $z = \max_k \lvert S_k\rvert$。前向きと後ろ向きの 2 つ | 2 |
| 2.14 | Random Excursions (`RandomExcursions`) | $\chi^2(x) = \sum_{k=0}^{5}\dfrac{(\nu_k(x) - J\pi_k(x))^2}{J\pi_k(x)}$、$P = \mathrm{igamc}(5/2, \chi^2/2)$ | $S_k$ が 0 に戻るまでの 1 周（$J$ 周）のうち、状態 $x = \pm1, \ldots, \pm4$ をちょうど $k$ 回（5 回以上をまとめる）訪れた周の数 $\nu_k(x)$ | 8 |
| 2.15 | Random Excursions Variant (`RandomExcursionsVariant`) | $P = \mathrm{erfc}\left(\dfrac{\lvert\xi(x) - J\rvert}{\sqrt{2J(4\lvert x\rvert - 2)}}\right)$ | 状態 $x = \pm1, \ldots, \pm9$ を訪れた総回数 $\xi(x)$ | 18 |
| | | | | **188** |
-->

| Section | Test (name in STS) | Test statistic and P-value | Brief description | Rows |
|---|---|---|---|---|
| 2.1 | Frequency (Monobit) (`Frequency`) | $s = \lvert\sum_i X_i\rvert/\sqrt{n}$, $P = \mathrm{erfc}(s/\sqrt{2})$ | Difference between the numbers of 0s and 1s | 1 |
| 2.2 | Frequency within a Block (`BlockFrequency`) | $\chi^2 = 4M\sum_{j=1}^{N}(\pi_j - \tfrac{1}{2})^2$, $P = \mathrm{igamc}(N/2, \chi^2/2)$ | Proportion of 1s $\pi_j$ in block $j$ of $M = 128$ bits ($N = \lfloor n/M \rfloor$) | 1 |
| 2.3 | Runs (`Runs`) | $P = \mathrm{erfc}\left(\dfrac{\lvert V - 2n\pi(1-\pi)\rvert}{2\sqrt{2n}\,\pi(1-\pi)}\right)$ | Number of runs of identical values $V$. $\pi$ is the proportion of 1s; $P = 0$ if $\lvert\pi - \tfrac{1}{2}\rvert > 2/\sqrt{n}$ | 1 |
| 2.4 | Longest Run of Ones in a Block (`LongestRun`) | $\chi^2 = \sum_{i=0}^{6}(\nu_i - N\pi_i)^2/(N\pi_i)$, $P = \mathrm{igamc}(3, \chi^2/2)$ | $\nu_i$: counts of the longest run of 1s in blocks of $M = 10^4$ bits, in 7 classes (≤10, 11, …, 15, ≥16) | 1 |
| 2.5 | Binary Matrix Rank (`Rank`) | $\chi^2 = \sum_{r}(F_r - Np_r)^2/(Np_r)$, $P = e^{-\chi^2/2}$ | Numbers $F_r$ of $32 \times 32$ matrices whose rank over GF(2) is 32, 31, or ≤30 ($p_{32} \approx 0.2888$, $p_{31} \approx 0.5776$, $p_{30} \approx 0.1336$) | 1 |
| 2.6 | Discrete Fourier Transform (`FFT`) | $d = \dfrac{N_1 - 0.95n/2}{\sqrt{n \cdot 0.95 \cdot 0.05/4}}$, $P = \mathrm{erfc}(\lvert d\rvert/\sqrt{2})$ | Number $N_1$ of discrete-Fourier-transform magnitudes of $X$ (first $n/2$) below $T = \sqrt{n\ln 20}$. Looks for periodic structure | 1 |
| 2.7 | Non-overlapping Template Matching (`NonOverlappingTemplate`) | $\chi^2 = \sum_{j=1}^{8}(W_j - \mu)^2/\sigma^2$, $P = \mathrm{igamc}(4, \chi^2/2)$ | Number of non-overlapping occurrences $W_j$ of an aperiodic $m = 9$-bit pattern in each of 8 blocks ($M = \lfloor n/8 \rfloor$). $\mu = (M - m + 1)/2^m$, $\sigma^2 = M(2^{-m} - (2m-1)2^{-2m})$ | 148 |
| 2.8 | Overlapping Template Matching (`OverlappingTemplate`) | $\chi^2 = \sum_{i=0}^{5}(\nu_i - N\pi_i)^2/(N\pi_i)$, $P = \mathrm{igamc}(5/2, \chi^2/2)$ | Numbers of blocks $\nu_i$ of $M = 1032$ bits by how many times (0, 1, …, 4, ≥5) a run of nine 1s occurs, overlaps allowed | 1 |
| 2.9 | Maurer's "Universal Statistical" (`Universal`) | $P = \mathrm{erfc}\left(\dfrac{\lvert f_n - \mathrm{E}[L]\rvert}{\sqrt{2}\,\sigma}\right)$ | Mean $f_n$ of $\log_2$ of the number of blocks since each $L = 7$-bit block last appeared (the first $10 \cdot 2^L$ blocks are used for initialization). Looks at compressibility | 1 |
| 2.10 | Linear Complexity (`LinearComplexity`) | $\chi^2 = \sum_{i=0}^{6}(\nu_i - N\pi_i)^2/(N\pi_i)$, $P = \mathrm{igamc}(3, \chi^2/2)$ | $\nu_i$: counts of $T_j = (-1)^M(L_j - \mu) + 2/9$ in 7 classes, from the linear complexity $L_j$ (Berlekamp–Massey algorithm) of blocks of $M = 500$ bits | 1 |
| 2.11 | Serial (`Serial`) | $P_1 = \mathrm{igamc}(2^{m-2}, \nabla\psi^2_m/2)$, $P_2 = \mathrm{igamc}(2^{m-3}, \nabla^2\psi^2_m/2)$ | From the frequencies $\nu$ of overlapping $m = 16$-bit patterns, $\psi^2_m = \frac{2^m}{n}\sum \nu^2 - n$, $\nabla\psi^2_m = \psi^2_m - \psi^2_{m-1}$, $\nabla^2\psi^2_m = \psi^2_m - 2\psi^2_{m-1} + \psi^2_{m-2}$ | 2 |
| 2.12 | Approximate Entropy (`ApproximateEntropy`) | $\chi^2 = 2n(\ln 2 - \mathrm{ApEn})$, $P = \mathrm{igamc}(2^{m-1}, \chi^2/2)$ | $\mathrm{ApEn} = \varphi^{(m)} - \varphi^{(m+1)}$, $\varphi^{(m)} = \sum_i \frac{C_i}{n}\ln\frac{C_i}{n}$ ($C_i$ is the count of overlapping $m = 10$-bit pattern $i$) | 1 |
| 2.13 | Cumulative Sums (`CumulativeSums`) | $P = 1 - \sum_k\bigl[\Phi(\tfrac{(4k+1)z}{\sqrt{n}}) - \Phi(\tfrac{(4k-1)z}{\sqrt{n}})\bigr] + \sum_k\bigl[\Phi(\tfrac{(4k+3)z}{\sqrt{n}}) - \Phi(\tfrac{(4k+1)z}{\sqrt{n}})\bigr]$ | Maximum excursion $z = \max_k \lvert S_k\rvert$ of the random walk $S_k = \sum_{i \le k} X_i$. Two variants: forward and backward | 2 |
| 2.14 | Random Excursions (`RandomExcursions`) | $\chi^2(x) = \sum_{k=0}^{5}\dfrac{(\nu_k(x) - J\pi_k(x))^2}{J\pi_k(x)}$, $P = \mathrm{igamc}(5/2, \chi^2/2)$ | Among the $J$ cycles of $S_k$ (each ending when it returns to 0), the number of cycles $\nu_k(x)$ that visit state $x = \pm1, \ldots, \pm4$ exactly $k$ times (5 or more grouped together) | 8 |
| 2.15 | Random Excursions Variant (`RandomExcursionsVariant`) | $P = \mathrm{erfc}\left(\dfrac{\lvert\xi(x) - J\rvert}{\sqrt{2J(4\lvert x\rvert - 2)}}\right)$ | Total number of visits $\xi(x)$ to state $x = \pm1, \ldots, \pm9$ | 18 |
| | | | | **188** |

<!-- 報告（`finalAnalysisReport.txt`）の行は、上の表の節の順ではなく、STS の内部の順（Frequency, BlockFrequency, CumulativeSums, Runs, LongestRun, Rank, FFT, NonOverlappingTemplate, OverlappingTemplate, Universal, ApproximateEntropy, RandomExcursions, RandomExcursionsVariant, Serial, LinearComplexity）に並びます。 -->

The rows of the report (`finalAnalysisReport.txt`) are not in the section order of the table above but in STS's internal order (Frequency, BlockFrequency, CumulativeSums, Runs, LongestRun, Rank, FFT, NonOverlappingTemplate, OverlappingTemplate, Universal, ApproximateEntropy, RandomExcursions, RandomExcursionsVariant, Serial, LinearComplexity).

<!-- ### C. 系列の集まりとしての判定（SP 800-22 の 4.2 節） -->

### C. Judging a collection of sequences (Section 4.2 of SP 800-22)

<!-- $s$ 本の系列に同じ検定を行い、報告の 1 行ごとに $s$ 個の P 値を次の 2 つの基準で調べます。基準を満たさない行には、STS が `*` をつけます。 -->

The same test is applied to $s$ sequences, and for each row of the report the $s$ P-values are checked against the following two criteria. STS marks rows that fail a criterion with `*`.

<!-- **基準 1：合格した系列の割合**。$\text{P 値} \ge \alpha$ の系列の割合が -->

**Criterion 1: proportion of passing sequences**. The criterion is met if the proportion of sequences with $\text{P-value} \ge \alpha$ falls within

$$
\hat{p} \pm 3\sqrt{\frac{\hat{p}(1-\hat{p})}{s}}, \qquad \hat{p} = 1 - \alpha
$$

<!-- の範囲に入れば合格です。STS は両端に $s$ を掛けて整数に切り捨てるので、$s = 55$ では **55 本中 52 本以上**が合格すれば基準を満たします。 -->

STS multiplies both ends by $s$ and truncates them to integers, so for $s = 55$ the criterion is met if **at least 52 of the 55** sequences pass.

<!-- **基準 2：P 値の一様性**。P 値を $[0, 0.1), \ldots, [0.9, 1]$ の 10 区間に分けた個数 $F_1, \ldots, F_{10}$（報告の `C1`〜`C10`）から -->

**Criterion 2: uniformity of the P-values**. From the counts $F_1, \ldots, F_{10}$ of P-values in the 10 intervals $[0, 0.1), \ldots, [0.9, 1]$ (`C1`–`C10` in the report), compute

$$
\chi^2 = \sum_{i=1}^{10}\frac{(F_i - s/10)^2}{s/10}, \qquad \text{P-value}_T = \mathrm{igamc}\left(\frac{9}{2}, \frac{\chi^2}{2}\right)
$$

<!-- を計算し、$\text{P-value}_T \ge 0.0001$ なら合格です。 -->

The criterion is met if $\text{P-value}_T \ge 0.0001$.

<!-- ### 但し書き -->

### Caveats

<!--
- **統計的検定は予測できないことの証明ではありません**：決まった計算で作る擬似乱数も合格します。QRNG の安全性の根拠は、SP 800-90B で評価した min-entropy と抽出器の理論です。SP 800-22 は、抽出器の出力に残った偏りや実装の誤りを見つけるために使います。
- **完全な乱数でも `*` はつきます**：$s = 55$ では、完全な乱数でも 1 行あたり基準 1 で約 0.23%、基準 2 で約 0.05% の確率で `*` がつき、188 行では平均 0.5 行程度です。行どうしは同じ系列から計算するので、数行に `*` がつくこともあります。`*` が少数で特定の検定に集中していなければ「乱数らしくない性質は見つからなかった」と読み、同じ検定に `*` が続く場合や、データを取り直しても同じ行に `*` が残る場合はビット列を疑います。
- **系列の長さと本数**：$10^6$ ビットの系列を 55 本以上使うのが標準です。系列が短いと、一部の検定のパラメータが変わるか、検定できなくなります。
- **Random Excursions の 2 つの検定**：$S_k$ が 0 に戻る回数 $J$ が $\max(0.005\sqrt{n}, 500)$ 未満の系列には適用できず、STS はその系列の P 値を 0 と書いて、基準の計算から除きます。$10^6$ ビットの乱数では約 6 割の系列が条件を満たすので、この 2 つの検定の行は $s$ が 55 より小さくなります。
- **STS の基準 2 の計算**：期待度数 $s/10$ を整数の割り算で計算するので（$s = 55$ なら 5）、$s$ が 10 の倍数でないと判定が少し厳しくなります。$s < 10$ では計算されず `----` と表示されます。
- **入力ファイルの形式**：STS には文字の `'0'` と `'1'` を区切らずに並べたテキストを渡します（SP 800-90B のツールとは逆の形式です）。STS はパスを長さ 200 の配列に読むので、パスは空白を含まない 199 文字以内にしてください。
-->

- **Statistical tests do not prove unpredictability**: pseudorandom numbers produced by a fixed computation also pass. The security of a QRNG rests on the min-entropy assessed with SP 800-90B and on the theory of extractors. SP 800-22 is used to find bias left in the extractor output and implementation errors.
- **Even perfect random numbers get `*`**: with $s = 55$, a row of perfect random numbers gets `*` with probability about 0.23% under criterion 1 and about 0.05% under criterion 2, which is about 0.5 rows on average over 188 rows. Because the rows are computed from the same sequences, several rows may get `*` together. If there are only a few `*` and they are not concentrated in a particular test, read the result as "no non-random property was found"; if `*` keeps appearing in the same test, or remains in the same row after the data are taken again, suspect the bit string.
- **Length and number of sequences**: the standard is to use at least 55 sequences of $10^6$ bits. With shorter sequences, some tests change their parameters or cannot be run.
- **The two Random Excursions tests**: they cannot be applied to sequences in which the number of returns to 0, $J$, is less than $\max(0.005\sqrt{n}, 500)$; STS writes a P-value of 0 for such sequences and excludes them from the criteria. For $10^6$-bit random sequences about 60% satisfy the condition, so the rows of these two tests have $s$ smaller than 55.
- **STS's computation of criterion 2**: the expected count $s/10$ is computed by integer division (5 for $s = 55$), so the judgment is slightly stricter when $s$ is not a multiple of 10. For $s < 10$ it is not computed and `----` is shown.
- **Input file format**: STS takes a text file of the characters `'0'` and `'1'` with no separators (the opposite of the format for the SP 800-90B tool). STS reads the path into an array of length 200, so the path must be at most 199 characters with no whitespace.

---

<!-- ## 使い方 -->

## How to use this notebook

<!-- まず `README.md` の手順で Python の環境を作り、このノートブックと同じフォルダに STS をビルドしてから、上のセルから順に実行してください。別の場所に置いた場合は `STS_DIR` を書き換えてください。条件は、次の変数を書き換えて選びます。 -->

First set up the Python environment and build STS in the same folder as this notebook by following `README.md`, then run the cells in order from the top. If you placed STS elsewhere, edit `STS_DIR`. The conditions are selected by editing the following variables.

<!--
| 変数 | 意味 | 既定値 |
|---|---|---|
| `STS_DIR` | STS のフォルダ（`assess`, `experiments`, `templates` があるフォルダ） | `sts-2.1.2/sts-2.1.2`（このノートブックからの相対パス） |
| `SAVE_RESULTS` | 結果を保存するか（`True` なら `results/sp800_22/<日時>/` に保存） | `False` |
| `BITS_FILE` | 検定するビット列のファイル（`None` ならシミュレーションを使う） | `None` |
| `N_SEQUENCES` | 系列の本数 $s$（`BITS_FILE` を使う場合は、ファイルから作れる本数が上限） | `55` |
| `SEQUENCE_LENGTH` | 1 本の系列のビット数 $n$ | `1_000_000` |
| `GATE` | C1 回路のゲート（`"H"`, `"Rx"`, `"Ry"`） | `"H"` |
| `READOUT_P1_GIVEN_0`, `READOUT_P0_GIVEN_1` | 読み出し誤差 $P(1 \mid 0)$, $P(0 \mid 1)$ | `0.03`, `0.03` |
| `SIM_SEED` | シミュレータの乱数シード（系列 $i$ には `SIM_SEED + i` を使う。`None` なら毎回違う結果） | `1` |
-->

| Variable | Meaning | Default |
|---|---|---|
| `STS_DIR` | STS folder (the folder containing `assess`, `experiments`, and `templates`) | `sts-2.1.2/sts-2.1.2` (relative to this notebook) |
| `SAVE_RESULTS` | Whether to save the results (`True` saves them in `results/sp800_22/<date_time>/`) | `False` |
| `BITS_FILE` | File of the bit string to test (`None` means use the simulation) | `None` |
| `N_SEQUENCES` | Number of sequences $s$ (when `BITS_FILE` is used, at most the number that can be made from the file) | `55` |
| `SEQUENCE_LENGTH` | Number of bits $n$ per sequence | `1_000_000` |
| `GATE` | Gate of the C1 circuit (`"H"`, `"Rx"`, `"Ry"`) | `"H"` |
| `READOUT_P1_GIVEN_0`, `READOUT_P0_GIVEN_1` | Readout errors $P(1 \mid 0)$, $P(0 \mid 1)$ | `0.03`, `0.03` |
| `SIM_SEED` | Random seed of the simulator (sequence $i$ uses `SIM_SEED + i`; `None` gives different results each time) | `1` |

<!-- **実行時間の目安**：$10^6$ ビットの系列 55 本で、シミュレーションに 1〜2 分、STS に 3〜5 分ほどかかります。STS は自分のフォルダの `experiments/AlgorithmTesting` に結果を上書きするので、同じ `STS_DIR` で 2 つの検定を同時に実行しないでください。 -->

**Approximate run time**: for 55 sequences of $10^6$ bits, the simulation takes 1–2 minutes and STS takes about 3–5 minutes. STS overwrites its results in `experiments/AlgorithmTesting` in its own folder, so do not run two tests at the same time with the same `STS_DIR`.

<!-- ### 保存されるファイル -->

### Saved files

<!-- 2. の準備のセルで `SAVE_RESULTS = True` にすると、そのセルを実行した日時の名前のフォルダ `results/sp800_22/<年-月-日_時分秒>/`（変数 `RUN_DIR`）を作り、次のファイルを保存します。`False`（既定）では何も保存しません。STS に渡す入力ファイル（全系列を 1 つにつなげたもの）は、どちらの場合も一時フォルダ `DATA_DIR` に書き、`RUN_DIR` には系列ごとのファイルを保存します。 -->

If you set `SAVE_RESULTS = True` in the setup cell in 2., a folder named after the date and time that cell was run, `results/sp800_22/<year-month-day_hourminutesecond>/` (variable `RUN_DIR`), is created and the following files are saved in it. With `False` (the default), nothing is saved. In either case, the input file for STS (all sequences joined into one file) is written to the temporary folder `DATA_DIR`; `RUN_DIR` holds one file per sequence instead.

<!--
```
results/sp800_22/
└── 2026-11-10_153012/                RUN_DIR（2. のセルを実行した日時）
    ├── sequences/                       検定した系列（1 本 1 ファイル）
    │   ├── c1_H_sim_seq001.txt
    │   ├── c1_H_sim_seq002.txt
    │   └── ...
    ├── c1_H_sim_sts_input.json          STS に渡した入力ファイルの記録
    ├── sts_results/                     STS の結果のフォルダのコピー
    │   ├── finalAnalysisReport.txt
    │   ├── freq.txt
    │   ├── Frequency/
    │   └── ...
    └── c1_H_sim_sts.csv                 報告の 188 行
```
-->

```
results/sp800_22/
└── 2026-11-10_153012/                RUN_DIR (date and time the cell in 2. was run)
    ├── sequences/                       tested sequences (one file per sequence)
    │   ├── c1_H_sim_seq001.txt
    │   ├── c1_H_sim_seq002.txt
    │   └── ...
    ├── c1_H_sim_sts_input.json          record of the input file passed to STS
    ├── sts_results/                     copy of the STS results folder
    │   ├── finalAnalysisReport.txt
    │   ├── freq.txt
    │   ├── Frequency/
    │   └── ...
    └── c1_H_sim_sts.csv                 the 188 rows of the report
```

<!-- ファイル名の先頭の `<data_label>` は、シミュレーションなら `c1_<GATE>_sim`（例：`c1_H_sim`）、`BITS_FILE` を使う場合はそのファイル名から拡張子を除いたもの（例：`toeplitz_extracted`）です。 -->

`<data_label>` at the start of each file name is `c1_<GATE>_sim` (e.g., `c1_H_sim`) for the simulation, or the file name of `BITS_FILE` without its extension (e.g., `toeplitz_extracted`) when `BITS_FILE` is used.

<!--
| ファイル | 形式 | 中身 | 書き出すセル |
|---|---|---|---|
| `sequences/<data_label>_seq<番号>.txt` | テキスト。`'0'` と `'1'` の文字を改行なしで並べる（1 文字 = 1 ビット） | 検定した系列の 1 本（$n$ ビット）。番号は 1 から数えた 3 桁で、STS の結果の各系列の順番と同じ。`load_bits` で読める | 3.3 |
| `<data_label>_sts_input.json` | JSON | STS に渡した入力ファイルの SHA-256、系列の本数と長さ、元のデータ（`BITS_FILE` か、シミュレーションの条件） | 3.3 |
| `sts_results/finalAnalysisReport.txt` | テキスト（STS の書式） | STS の報告。188 行の `C1`〜`C10`、$\text{P-value}_T$、合格した系列の数 / 系列の数、`*` と、基準 1 の最小の合格数 | 4.2 |
| `sts_results/<検定>/results.txt` | テキスト（1 行に P 値 1 つ） | その検定の全系列の P 値。1 本の系列で複数の P 値を出す検定は、その全部（NonOverlappingTemplate なら 148 × 55 行） | 4.2 |
| `sts_results/<検定>/data<k>.txt` | テキスト（1 行に P 値 1 つ） | 1 本の系列で複数の P 値を出す検定（CumulativeSums, NonOverlappingTemplate, RandomExcursions, RandomExcursionsVariant, Serial）だけにある、報告の $k$ 行目の各系列の P 値（$s$ 行）。Random Excursions の 2 つの検定で適用できない系列は 0 | 4.2 |
| `sts_results/<検定>/stats.txt` | テキスト（STS の書式） | 系列ごとの検定統計量などの計算の詳細 | 4.2 |
| `sts_results/freq.txt` | テキスト（STS の書式） | 系列ごとの 0 と 1 の個数 | 4.2 |
| `<data_label>_sts.csv` | CSV（ヘッダー 1 行 + 報告の 1 行につき 1 行） | 報告の 188 行。列は下の表 | 5. |
-->

| File | Format | Contents | Written by cell |
|---|---|---|---|
| `sequences/<data_label>_seq<number>.txt` | Text. The characters `'0'` and `'1'` with no line breaks (1 character = 1 bit) | One tested sequence ($n$ bits). The number is counted from 1 (3 digits) and matches the order of the sequences in the STS results. Can be read with `load_bits` | 3.3 |
| `<data_label>_sts_input.json` | JSON | SHA-256 of the input file passed to STS, the number and length of the sequences, and the source data (`BITS_FILE` or the simulation conditions) | 3.3 |
| `sts_results/finalAnalysisReport.txt` | Text (STS format) | The STS report: `C1`–`C10`, $\text{P-value}_T$, passing sequences / sequences, and `*` for the 188 rows, plus the minimum pass count for criterion 1 | 4.2 |
| `sts_results/<test>/results.txt` | Text (one P-value per line) | P-values of all sequences for the test. For tests that give several P-values per sequence, all of them (148 × 55 lines for NonOverlappingTemplate) | 4.2 |
| `sts_results/<test>/data<k>.txt` | Text (one P-value per line) | Only for tests that give several P-values per sequence (CumulativeSums, NonOverlappingTemplate, RandomExcursions, RandomExcursionsVariant, Serial): the P-value of each sequence for row $k$ of the report ($s$ lines). 0 for sequences to which the two Random Excursions tests cannot be applied | 4.2 |
| `sts_results/<test>/stats.txt` | Text (STS format) | Details of the computation, such as the test statistic for each sequence | 4.2 |
| `sts_results/freq.txt` | Text (STS format) | Numbers of 0s and 1s in each sequence | 4.2 |
| `<data_label>_sts.csv` | CSV (1 header row + 1 row per report row) | The 188 rows of the report. Columns are listed in the table below | 5. |

<!-- `<data_label>_sts.csv` の列は次のとおりです。報告で `----` や `------` と表示される値は空欄です。 -->

The columns of `<data_label>_sts.csv` are as follows. Values shown as `----` or `------` in the report are left empty.

<!--
| 列 | 意味 |
|---|---|
| `data` | `<data_label>` |
| `test` | 検定の名前（STS の名前） |
| `row` | その検定の中で何行目か（1 から） |
| `variant` | その行が表すもの（テンプレート `B = 000000001`、状態 `x = -4`、`forward` など）。1 行だけの検定は空欄 |
| `C1`〜`C10` | P 値を $[0, 0.1), \ldots, [0.9, 1]$ の 10 区間に分けた系列の数 |
| `uniformity_p` | 基準 2 の $\text{P-value}_T$ |
| `uniformity_flag` | 基準 2 を満たさなければ `1`、満たせば `0` |
| `passed`, `total` | 基準 1 の、合格した（$\text{P 値} \ge \alpha$ の）系列の数と、系列の数 |
| `proportion_flag` | 基準 1 を満たさなければ `1`、満たせば `0` |
| `generator` | STS が読んだ入力ファイルの絶対パス |
-->

| Column | Meaning |
|---|---|
| `data` | `<data_label>` |
| `test` | Name of the test (name in STS) |
| `row` | Row number within the test (from 1) |
| `variant` | What the row represents (template `B = 000000001`, state `x = -4`, `forward`, and so on). Empty for tests with only one row |
| `C1`–`C10` | Numbers of sequences whose P-value falls in each of the 10 intervals $[0, 0.1), \ldots, [0.9, 1]$ |
| `uniformity_p` | $\text{P-value}_T$ of criterion 2 |
| `uniformity_flag` | `1` if criterion 2 is not met, `0` if it is |
| `passed`, `total` | For criterion 1, the number of passing sequences (with $\text{P-value} \ge \alpha$) and the number of sequences |
| `proportion_flag` | `1` if criterion 1 is not met, `0` if it is |
| `generator` | Absolute path of the input file read by STS |

<!--
- ファイルは上書きしません。同じ `RUN_DIR` のまま 3.3・4.2・5. のセルをもう一度実行するとエラー（`FileExistsError`）になるので、2. の準備のセルを実行し直して新しいフォルダを作ってください。
- `<data_label>_sts.csv` の `generator` 列は、STS が読んだ一時フォルダの入力ファイルのパスです。入力ファイルは `<data_label>_sts_input.json` の SHA-256 で照合できます。
-->

- Files are never overwritten. If you run the cell in 3.3, 4.2, or 5. again with the same `RUN_DIR`, an error (`FileExistsError`) is raised, so rerun the setup cell in 2. to create a new folder.
- The `generator` column of `<data_label>_sts.csv` is the path of the input file in the temporary folder read by STS. The input file can be checked against the SHA-256 in `<data_label>_sts_input.json`.

<!-- ### 関数の家系図 -->

### Function call tree

<!-- このノートブックの関数の呼び出し関係です。上の（字下げの浅い）関数が、そのすぐ下に枝で続く関数を内部で呼び出します。 -->

This shows how the functions in this notebook call each other. A function higher up (less indented) internally calls the functions on the branches directly below it.

<!--
- `[3.1]` のような右の番号は、**その関数が定義されている章**です。
- `← 4.2 のセルから実行` は、その章のコードセルから直接呼び出していることを示します。
- `（↑ 上で展開済み）` は、同じ関数の中身をすでに上の図で展開していることを示します。
-->

- The number on the right, such as `[3.1]`, is **the section in which the function is defined**.
- `← run from the cell in 4.2` means the function is called directly from the code cell of that section.
- `(↑ expanded above)` means the contents of the same function have already been expanded earlier in the tree.

<!-- **検定する系列の準備** -->

**Preparing the sequences to test**

<!--
```
c1_sequences          [3.1]  ← 3.3 のセルから実行
└── simulate_c1_bits  [3.1]
    └── build_c1      [3.1]

build_noise_model     [3.1]  ← 3.3 のセルから実行
load_bits             [3.2]  ← 3.3 のセルから実行
split_sequences       [3.2]  ← 3.3 のセルから実行

save_sequences        [3.2]  ← 3.3 のセルから実行
write_sts_input       [3.2]  ← 3.3 のセルから実行
└── check_sts_path    [3.2]
```
-->

```
c1_sequences          [3.1]  ← run from the cell in 3.3
└── simulate_c1_bits  [3.1]
    └── build_c1      [3.1]

build_noise_model     [3.1]  ← run from the cell in 3.3
load_bits             [3.2]  ← run from the cell in 3.3
split_sequences       [3.2]  ← run from the cell in 3.3

save_sequences        [3.2]  ← run from the cell in 3.3
write_sts_input       [3.2]  ← run from the cell in 3.3
└── check_sts_path    [3.2]
```

<!-- **STS の実行と結果の読み取り・保存** -->

**Running STS and reading and saving the results**

<!--
```
sts_available           [2.]  ← 2. のセルから実行

run_sts                 [4.1]  ← 4.2 のセルから実行
└── check_sts_path      [3.2]

parse_sts_report        [4.1]  ← 4.2 のセルから実行
show_sts_summary        [4.1]  ← 4.2 のセルから実行

show_flagged_rows       [4.1]  ← 4.2 のセルから実行
└── row_label           [4.1]
    └── variant_labels  [4.1]

save_sts_csv            [5.]  ← 5. のセルから実行
└── row_label           [4.1]  （↑ 上で展開済み）
```
-->

```
sts_available           [2.]  ← run from the cell in 2.

run_sts                 [4.1]  ← run from the cell in 4.2
└── check_sts_path      [3.2]

parse_sts_report        [4.1]  ← run from the cell in 4.2
show_sts_summary        [4.1]  ← run from the cell in 4.2

show_flagged_rows       [4.1]  ← run from the cell in 4.2
└── row_label           [4.1]
    └── variant_labels  [4.1]

save_sts_csv            [5.]  ← run from the cell in 5.
└── row_label           [4.1]  (↑ expanded above)
```

<!-- **章ごとの定義の一覧** -->

**Functions defined in each section**

<!--
| 章 | 章の題 | 定義されている関数 |
|---|---|---|
| 2. | 準備 | `sts_available` |
| 3.1 | C1 回路のシミュレーションの関数 | `build_c1`、`build_noise_model`、`simulate_c1_bits`、`c1_sequences` |
| 3.2 | ファイルを扱う関数 | `load_bits`、`split_sequences`、`check_sts_path`、`write_sts_input`、`save_sequences` |
| 4.1 | STS を実行して報告を読む関数 | `run_sts`、`parse_sts_report`、`variant_labels`、`row_label`、`show_sts_summary`、`show_flagged_rows` |
| 5. | 結果を保存する | `save_sts_csv` |
-->

| Section | Section title | Functions defined |
|---|---|---|
| 2. | Setup | `sts_available` |
| 3.1 | Functions for simulating the C1 circuit | `build_c1`, `build_noise_model`, `simulate_c1_bits`, `c1_sequences` |
| 3.2 | Functions for handling files | `load_bits`, `split_sequences`, `check_sts_path`, `write_sts_input`, `save_sequences` |
| 4.1 | Functions for running STS and reading the report | `run_sts`, `parse_sts_report`, `variant_labels`, `row_label`, `show_sts_summary`, `show_flagged_rows` |
| 5. | Saving the results | `save_sts_csv` |

<!-- ## 1. 公式ツール STS -->

## 1. The official tool STS

<!--
- 配布ページ：<https://csrc.nist.gov/projects/random-bit-generation/documentation-and-software>（"NIST SP 800-22 Rev. 1a" の項目の `sts-2_1_2.zip`）
- SP 800-22 Rev. 1a の本文：<https://nvlpubs.nist.gov/nistpubs/Legacy/SP/nistspecialpublication800-22r1a.pdf>
-->

- Distribution page: <https://csrc.nist.gov/projects/random-bit-generation/documentation-and-software> (`sts-2_1_2.zip` under "NIST SP 800-22 Rev. 1a")
- Text of SP 800-22 Rev. 1a: <https://nvlpubs.nist.gov/nistpubs/Legacy/SP/nistspecialpublication800-22r1a.pdf>

<!-- STS のダウンロードとビルドは、リポジトリの `README.md` の手順で行います。その手順どおりにすると、STS はこのノートブックと同じフォルダの `sts-2.1.2/sts-2.1.2` にビルドされます（SHA-256 が `0238d2f1d26e120e3cc748ed2d4c674cdc636de37fc4027c76cc2a394fff9157` の `sts-2_1_2.zip` で確認しました）。このフォルダ（`assess`, `experiments`, `templates` があるフォルダ）が `STS_DIR` です。 -->

Download and build STS by following `README.md` in the repository. If you follow those steps, STS is built in `sts-2.1.2/sts-2.1.2` in the same folder as this notebook (verified with the `sts-2_1_2.zip` whose SHA-256 is `0238d2f1d26e120e3cc748ed2d4c674cdc636de37fc4027c76cc2a394fff9157`). This folder (the one containing `assess`, `experiments`, and `templates`) is `STS_DIR`.

<!-- **`assess` の使い方** -->

**How to use `assess`**

<!-- STS は必ず `STS_DIR` の中で `./assess <系列のビット数 n>` として実行します。次の質問に答えると検定が始まります。4. の `run_sts` は、同じ答えを標準入力から渡します。 -->

STS must always be run inside `STS_DIR` as `./assess <number of bits per sequence n>`. The tests start after you answer the following questions. `run_sts` in 4. passes the same answers through standard input.

<!--
| 質問 | 答え | 意味 |
|---|---|---|
| `Enter Choice:`（GENERATOR SELECTION） | `0` | ファイルを読む |
| `User Prescribed Input File:` | ファイルの絶対パス | 空白を含まない 199 文字以内 |
| `Enter Choice:`（STATISTICAL TESTS） | `1` | 15 種類すべての検定 |
| `Select Test (0 to continue):` | `0` | パラメータは既定値のまま |
| `How many bitstreams?` | `55` | 系列の本数 $s$ |
| `Select input mode:` | `0` | ASCII 形式（`'0'` と `'1'` の文字） |
-->

| Question | Answer | Meaning |
|---|---|---|
| `Enter Choice:` (GENERATOR SELECTION) | `0` | Read from a file |
| `User Prescribed Input File:` | Absolute path of the file | At most 199 characters with no whitespace |
| `Enter Choice:` (STATISTICAL TESTS) | `1` | All 15 tests |
| `Select Test (0 to continue):` | `0` | Keep the default parameters |
| `How many bitstreams?` | `55` | Number of sequences $s$ |
| `Select input mode:` | `0` | ASCII format (the characters `'0'` and `'1'`) |

<!-- 結果は `STS_DIR/experiments/AlgorithmTesting/` に書かれます。`finalAnalysisReport.txt` が 188 行の報告、`<検定>/results.txt` が各系列の P 値、`<検定>/stats.txt` が検定統計量などの詳細です。 -->

The results are written to `STS_DIR/experiments/AlgorithmTesting/`. `finalAnalysisReport.txt` is the 188-row report, `<test>/results.txt` holds the P-value of each sequence, and `<test>/stats.txt` holds details such as the test statistics.

<!-- ## 2. 準備 -->

## 2. Setup

<!-- ライブラリを読み込み、STS の場所 `STS_DIR` と、結果を保存するかどうか `SAVE_RESULTS` を指定します。STS が見つからない場合は、STS を使うセルは実行せず、そのことを表示します。 -->

We load the libraries and specify the location of STS, `STS_DIR`, and whether to save the results, `SAVE_RESULTS`. If STS is not found, the cells that use STS are skipped and a message says so.

<!-- 結果の保存：`SAVE_RESULTS` で、検定した系列と STS の結果を保存するかどうかを選びます。`True` にすると、このセルを実行した日時の名前のフォルダ（例：`results/sp800_22/2026-11-10_153012/`）を作り、その中に保存します。`False`（既定）では何も保存しません。STS に渡す入力ファイルは、どちらの場合も一時フォルダ `DATA_DIR` に書きます。 -->

Saving the results: `SAVE_RESULTS` selects whether the tested sequences and the STS results are saved. With `True`, a folder named after the date and time this cell was run (e.g., `results/sp800_22/2026-11-10_153012/`) is created and the files are saved in it. With `False` (the default), nothing is saved. In either case, the input file for STS is written to the temporary folder `DATA_DIR`.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `sts_available` | STS のフォルダに実行に必要なもの（`assess`、結果のフォルダ、`templates`）がそろっているかを判定することを目的とした関数です。 | `sts_dir` に `assess`、`experiments/AlgorithmTesting`、`templates` があれば `True` | `sts_available(STS_DIR)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `sts_available` | Checks whether the STS folder has everything needed to run (`assess`, the results folder, and `templates`). | `True` if `sts_dir` contains `assess`, `experiments/AlgorithmTesting`, and `templates` | `sts_available(STS_DIR)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>sts_available</code></td><td><code>sts_dir</code></td><td>STS のフォルダ</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>sts_available</code></td><td><code>sts_dir</code></td><td>STS folder</td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
import csv
import json
import hashlib
import math
import re
import shutil
import subprocess
import tempfile
import time
from datetime import datetime
from pathlib import Path

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error, ReadoutError
from IPython.display import Markdown, display

# STS folder built by following README.md (extracted into the same folder as this notebook)
STS_DIR = Path("sts-2.1.2/sts-2.1.2").resolve()

# True: save the files of this run in a new folder results/sp800_22/<date_time>/; False: save nothing
SAVE_RESULTS = False
RESULTS_ROOT = Path("results/sp800_22")
RUN_DIR = RESULTS_ROOT / datetime.now().strftime("%Y-%m-%d_%H%M%S") if SAVE_RESULTS else None
print("save to    :", RUN_DIR if RUN_DIR is not None else "nothing is saved (SAVE_RESULTS = False)")
# Folder for the input file of STS (always a temporary folder; the sequences are saved one file each in RUN_DIR)
DATA_DIR = Path(tempfile.mkdtemp(prefix="sp800_22_"))


def sts_available(sts_dir):
    sts_dir = Path(sts_dir)
    return ((sts_dir / "assess").is_file()
            and (sts_dir / "experiments" / "AlgorithmTesting").is_dir()
            and (sts_dir / "templates").is_dir())


STS_AVAILABLE = sts_available(STS_DIR)
if STS_AVAILABLE:
    print("STS:", STS_DIR)
else:
    print(f"STS not found: {STS_DIR}")
    print("Build it in the same folder as this notebook by following README.md, or edit STS_DIR. Cells that use STS will be skipped.")

<!-- ## 3. 検定する系列を STS の入力ファイルに書く -->

## 3. Writing the sequences to test to the STS input file

<!-- ### 3.1 C1 回路のシミュレーションの関数 -->

### 3.1 Functions for simulating the C1 circuit

<!-- C1 回路（$|0\rangle$ に $H$, $R_X(\pi/2)$, $R_Y(\pi/2)$ のいずれかを作用させて測定する回路）を Qiskit Aer でシミュレーションします。回路はネイティブゲート $R_X$, $R_Y$, $\mathrm{CZ}$ に変換し（`optimization_level=0`）、次のノイズを加えます。 -->

We simulate the C1 circuit (a circuit that applies one of $H$, $R_X(\pi/2)$, $R_Y(\pi/2)$ to $|0\rangle$ and measures) with Qiskit Aer. The circuit is converted to the native gates $R_X$, $R_Y$, $\mathrm{CZ}$ (`optimization_level=0`), and the following noise is added.

<!--
| ノイズ | 値 |
|---|---|
| 1 量子ビットゲートの脱分極誤差 | $\lambda = 0.00198$ |
| 1 量子ビットゲートの熱緩和誤差 | $T_1 = 0.964$ ms, $T_2 = 1.155$ ms, ゲート時間 20 ns |
| 読み出し誤差 | $P(1 \mid 0)$, $P(0 \mid 1)$（`READOUT_P1_GIVEN_0`, `READOUT_P0_GIVEN_1`） |
-->

| Noise | Value |
|---|---|
| Depolarizing error of 1-qubit gates | $\lambda = 0.00198$ |
| Thermal relaxation error of 1-qubit gates | $T_1 = 0.964$ ms, $T_2 = 1.155$ ms, gate time 20 ns |
| Readout error | $P(1 \mid 0)$, $P(0 \mid 1)$ (`READOUT_P1_GIVEN_0`, `READOUT_P0_GIVEN_1`) |

<!-- $5.5 \times 10^7$ 回の測定結果を一度に受け取るとメモリが足りなくなるので、`c1_sequences` で系列を 1 本ずつ作ります。系列 $i$ にはシード `seed + i` を使います（同じシードでは同じ系列になるため）。 -->

Receiving $5.5 \times 10^7$ measurement results at once would run out of memory, so `c1_sequences` generates the sequences one at a time. Sequence $i$ uses the seed `seed + i` (because the same seed gives the same sequence).

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `build_c1` | C1 回路（1 量子ビットに重ね合わせを作るゲートを 1 個作用させて測定する回路）を作ることを目的とした関数です。 | 測定つきの `QuantumCircuit` | `build_c1("H")` |
| `build_noise_model` | 実機を模したシミュレーションのために、ゲートノイズと読み出し誤差を含むノイズモデルを作ることを目的とした関数です。 | `NoiseModel` | `build_noise_model(0.03, 0.03)` |
| `simulate_c1_bits` | C1 回路を Qiskit でシミュレーションし、検定する系列となる測定順のビット列を得ることを目的とした関数です。 | 測定順のビット列（長さ `shots` の `np.uint8` の配列） | `simulate_c1_bits("H", 1000, build_noise_model(0.03, 0.03), seed=1)` |
| `c1_sequences` | C1 回路のシミュレーションを繰り返し、検定する系列を一度にメモリに持たずに 1 本ずつ順に作ることを目的とした関数です。 | 長さ `seq_length` のビット列を `n_sequences` 本、順に返すジェネレータ | `list(c1_sequences(2, 10, "H", seed=1))` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `build_c1` | Builds the C1 circuit (a circuit that applies one superposition-creating gate to one qubit and measures it). | `QuantumCircuit` with measurement | `build_c1("H")` |
| `build_noise_model` | Builds a noise model with gate noise and readout errors, for a simulation that mimics real hardware. | `NoiseModel` | `build_noise_model(0.03, 0.03)` |
| `simulate_c1_bits` | Simulates the C1 circuit with Qiskit and obtains a bit string in measurement order to be used as a test sequence. | Bit string in measurement order (`np.uint8` array of length `shots`) | `simulate_c1_bits("H", 1000, build_noise_model(0.03, 0.03), seed=1)` |
| `c1_sequences` | Repeats the C1-circuit simulation and generates the test sequences one at a time, without holding them all in memory. | Generator that yields `n_sequences` bit strings of length `seq_length` in order | `list(c1_sequences(2, 10, "H", seed=1))` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>build_c1</code></td><td><code>gate</code></td><td><code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td rowspan="3"><code>build_noise_model</code></td><td><code>p1_given_0</code></td><td>$P(1|0)$</td><td><code>0.0</code></td></tr>
<tr><td><code>p0_given_1</code></td><td>$P(0|1)$</td><td><code>0.0</code></td></tr>
<tr><td><code>gate_noise</code></td><td>ゲートノイズ（脱分極 + 熱緩和）を加えるなら <code>True</code></td><td><code>True</code></td></tr>
<tr><td rowspan="4"><code>simulate_c1_bits</code></td><td><code>gate</code></td><td><code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>必須</td></tr>
<tr><td><code>shots</code></td><td>測定回数</td><td>必須</td></tr>
<tr><td><code>noise_model</code></td><td><code>NoiseModel</code>。<code>None</code> ならノイズなし</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>シミュレータの乱数シード（<code>None</code> なら毎回違う結果）</td><td><code>None</code></td></tr>
<tr><td rowspan="5"><code>c1_sequences</code></td><td><code>n_sequences</code></td><td>系列の本数</td><td>必須</td></tr>
<tr><td><code>seq_length</code></td><td>1 本の系列のビット数</td><td>必須</td></tr>
<tr><td><code>gate</code></td><td><code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td><code>"H"</code></td></tr>
<tr><td><code>noise_model</code></td><td><code>NoiseModel</code>。<code>None</code> ならノイズなし</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>最初の系列のシード。系列 $i$ には <code>seed + i</code> を使う</td><td><code>None</code></td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>build_c1</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td rowspan="3"><code>build_noise_model</code></td><td><code>p1_given_0</code></td><td>$P(1|0)$</td><td><code>0.0</code></td></tr>
<tr><td><code>p0_given_1</code></td><td>$P(0|1)$</td><td><code>0.0</code></td></tr>
<tr><td><code>gate_noise</code></td><td><code>True</code> to add gate noise (depolarizing + thermal relaxation)</td><td><code>True</code></td></tr>
<tr><td rowspan="4"><code>simulate_c1_bits</code></td><td><code>gate</code></td><td><code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>shots</code></td><td>Number of measurements</td><td>Required</td></tr>
<tr><td><code>noise_model</code></td><td><code>NoiseModel</code>. <code>None</code> means no noise</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>Random seed of the simulator (<code>None</code> gives different results each time)</td><td><code>None</code></td></tr>
<tr><td rowspan="5"><code>c1_sequences</code></td><td><code>n_sequences</code></td><td>Number of sequences</td><td>Required</td></tr>
<tr><td><code>seq_length</code></td><td>Number of bits per sequence</td><td>Required</td></tr>
<tr><td><code>gate</code></td><td><code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td><code>"H"</code></td></tr>
<tr><td><code>noise_model</code></td><td><code>NoiseModel</code>. <code>None</code> means no noise</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>Seed of the first sequence. Sequence $i$ uses <code>seed + i</code></td><td><code>None</code></td></tr>
</tbody>
</table>

In [ ]:
BASIS_GATES = ["rx", "ry", "cz"]
GATES = ["H", "Rx", "Ry"]
DEPOLARIZING_PARAM = 0.00198
T1, T2, GATE_TIME = 0.964e6, 1.155e6, 20   # ns


def build_c1(gate):
    if gate not in GATES:
        raise ValueError(f"gate must be one of {GATES} (got: {gate!r})")
    qc = QuantumCircuit(1, 1)
    if gate == "H":
        qc.h(0)
    elif gate == "Rx":
        qc.rx(math.pi / 2, 0)
    else:
        qc.ry(math.pi / 2, 0)
    qc.measure(0, 0)
    return qc


def build_noise_model(p1_given_0=0.0, p0_given_1=0.0, gate_noise=True):
    noise_model = NoiseModel(basis_gates=BASIS_GATES)
    if gate_noise:
        error = depolarizing_error(DEPOLARIZING_PARAM, 1).compose(thermal_relaxation_error(T1, T2, GATE_TIME))
        noise_model.add_all_qubit_quantum_error(error, ["rx", "ry"])
    if p1_given_0 or p0_given_1:
        noise_model.add_all_qubit_readout_error(ReadoutError([
            [1 - p1_given_0, p1_given_0],   # true value 0: P(0|0), P(1|0)
            [p0_given_1, 1 - p0_given_1],   # true value 1: P(0|1), P(1|1)
        ]))
    return noise_model


def simulate_c1_bits(gate, shots, noise_model=None, seed=None):
    qc_t = transpile(build_c1(gate), basis_gates=BASIS_GATES, optimization_level=0)
    backend = AerSimulator(noise_model=noise_model, seed_simulator=seed)
    memory = backend.run(qc_t, shots=shots, memory=True).result().get_memory()   # list of "0"/"1" in measurement order
    return (np.frombuffer("".join(memory).encode("ascii"), dtype=np.uint8) - ord("0")).astype(np.uint8)


def c1_sequences(n_sequences, seq_length, gate="H", noise_model=None, seed=None):
    for i in range(n_sequences):
        yield simulate_c1_bits(gate, seq_length, noise_model, seed=None if seed is None else seed + i)

<!-- ### 3.2 ファイルを扱う関数 -->

### 3.2 Functions for handling files

<!-- `write_sts_input` は、系列を 1 本ずつ受け取り、`'0'` と `'1'` の文字で区切らずにつなげて書きます。STS は先頭から $n$ ビットずつ読んで 1 本の系列とします。`save_sequences` は、`write_sts_input` に渡す系列をそのまま流しながら、1 本ずつ別のファイルに保存します。保存したビット列（Toeplitz・Trevisan のノートブックの `toeplitz_extracted.txt` など）は、`load_bits` で読んで `split_sequences` で系列に分けます。 -->

`write_sts_input` receives the sequences one at a time and writes them as the characters `'0'` and `'1'` joined with no separators. STS reads $n$ bits at a time from the start, each forming one sequence. `save_sequences` passes the sequences on to `write_sts_input` unchanged while saving each one as its own file. A saved bit string (such as `toeplitz_extracted.txt` from the Toeplitz or Trevisan notebook) is read with `load_bits` and split into sequences with `split_sequences`.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `load_bits` | 保存したビット列（抽出器の出力など）の 0 と 1 のテキストファイルを読み込み、検定の入力に使える配列に変換することを目的とした関数です。 | ビット列（`np.uint8` の配列） | `load_bits("results/toeplitz/2026-11-10_153012/toeplitz_extracted.txt")` |
| `split_sequences` | 1 本の長いビット列を、長さ `seq_length` の複数の系列に分けることを目的とした関数です。 | `(n_sequences, seq_length)` の 2 次元配列（余りは捨てる） | `split_sequences(bits, 1_000_000)` |
| `check_sts_path` | ファイルのパスが STS で読める形（空白を含まない 199 文字以内）であることを検査し、絶対パスに変換することを目的とした関数です。 | 絶対パスの文字列。空白を含むか 200 文字以上なら `ValueError` | `check_sts_path(DATA_DIR / "c1.txt")` |
| `write_sts_input` | 複数の系列を STS の入力形式（`'0'` と `'1'` の文字を区切らずにつなげたテキスト）で 1 つのファイルに書き出すことを目的とした関数です。 | `path`, `n_sequences`, `seq_length`, `n_bits`, `sha256` をキーとする辞書 | `write_sts_input(DATA_DIR / "test.txt", [[0, 1, 1], [1, 0, 0]])` |
| `save_sequences` | 系列を 1 本ずつ別のファイルに保存しながら、そのまま次の処理に渡すことを目的とした関数です。 | 受け取った系列を順に返すジェネレータ（`out_dir` に `<label>_seq001.txt` などを保存） | `write_sts_input(path, save_sequences(seqs, RUN_DIR / "sequences", "c1_H_sim"))` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `load_bits` | Reads a saved text file of 0s and 1s (such as the output of an extractor) and converts it into an array that can be used as test input. | Bit string (`np.uint8` array) | `load_bits("results/toeplitz/2026-11-10_153012/toeplitz_extracted.txt")` |
| `split_sequences` | Splits one long bit string into several sequences of length `seq_length`. | 2-D array of shape `(n_sequences, seq_length)` (the remainder is discarded) | `split_sequences(bits, 1_000_000)` |
| `check_sts_path` | Checks that a file path is in a form STS can read (at most 199 characters with no whitespace) and converts it to an absolute path. | Absolute path as a string. `ValueError` if it contains whitespace or is 200 characters or longer | `check_sts_path(DATA_DIR / "c1.txt")` |
| `write_sts_input` | Writes several sequences to one file in the STS input format (text of the characters `'0'` and `'1'` joined with no separators). | Dictionary with the keys `path`, `n_sequences`, `seq_length`, `n_bits`, and `sha256` | `write_sts_input(DATA_DIR / "test.txt", [[0, 1, 1], [1, 0, 0]])` |
| `save_sequences` | Saves each sequence as its own file while passing the sequences on unchanged to the next step. | Generator that yields the received sequences in order (saves `<label>_seq001.txt` and so on in `out_dir`) | `write_sts_input(path, save_sequences(seqs, RUN_DIR / "sequences", "c1_H_sim"))` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>load_bits</code></td><td><code>path</code></td><td><code>'0'</code> と <code>'1'</code> の文字のファイル（空白・改行は無視）</td><td>必須</td></tr>
<tr><td rowspan="3"><code>split_sequences</code></td><td><code>bits</code></td><td>ビット列</td><td>必須</td></tr>
<tr><td><code>seq_length</code></td><td>1 本の系列のビット数</td><td>必須</td></tr>
<tr><td><code>n_sequences</code></td><td>系列の本数。<code>None</code> なら作れるだけ</td><td><code>None</code></td></tr>
<tr><td><code>check_sts_path</code></td><td><code>path</code></td><td>ファイルのパス（まだ存在しなくてもよい）</td><td>必須</td></tr>
<tr><td rowspan="2"><code>write_sts_input</code></td><td><code>path</code></td><td>保存先。既に存在する場合は上書き</td><td>必須</td></tr>
<tr><td><code>sequences</code></td><td>同じ長さの 0 と 1 の系列を順に返すもの（リスト、2 次元配列、ジェネレータ）</td><td>必須</td></tr>
<tr><td rowspan="3"><code>save_sequences</code></td><td><code>sequences</code></td><td>系列を順に返すもの（リスト、2 次元配列、ジェネレータ）</td><td>必須</td></tr>
<tr><td><code>out_dir</code></td><td>保存先のフォルダ。<code>None</code> なら保存しない。同じ名前のファイルがあるとエラー</td><td>必須</td></tr>
<tr><td><code>label</code></td><td>ファイル名の先頭（例：<code>"c1_H_sim"</code>）</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>load_bits</code></td><td><code>path</code></td><td>File of the characters <code>'0'</code> and <code>'1'</code> (whitespace and newlines are ignored)</td><td>Required</td></tr>
<tr><td rowspan="3"><code>split_sequences</code></td><td><code>bits</code></td><td>Bit string</td><td>Required</td></tr>
<tr><td><code>seq_length</code></td><td>Number of bits per sequence</td><td>Required</td></tr>
<tr><td><code>n_sequences</code></td><td>Number of sequences. <code>None</code> means as many as possible</td><td><code>None</code></td></tr>
<tr><td><code>check_sts_path</code></td><td><code>path</code></td><td>File path (the file need not exist yet)</td><td>Required</td></tr>
<tr><td rowspan="2"><code>write_sts_input</code></td><td><code>path</code></td><td>Destination. Overwritten if it already exists</td><td>Required</td></tr>
<tr><td><code>sequences</code></td><td>Something that yields sequences of 0s and 1s of equal length in order (a list, a 2-D array, or a generator)</td><td>Required</td></tr>
<tr><td rowspan="3"><code>save_sequences</code></td><td><code>sequences</code></td><td>Something that yields the sequences in order (a list, a 2-D array, or a generator)</td><td>Required</td></tr>
<tr><td><code>out_dir</code></td><td>Destination folder. <code>None</code> means do not save. An error is raised if a file with the same name exists</td><td>Required</td></tr>
<tr><td><code>label</code></td><td>Start of the file names (e.g., <code>"c1_H_sim"</code>)</td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
def load_bits(path):
    text = "".join(Path(path).read_text(encoding="ascii").split())
    bits = np.frombuffer(text.encode("ascii"), dtype=np.uint8) - ord("0")
    if bits.size == 0 or np.any(bits > 1):
        raise ValueError(f"the file contains characters other than 0 and 1, or is empty: {path}")
    return bits.astype(np.uint8)


def split_sequences(bits, seq_length, n_sequences=None):
    bits = np.asarray(bits, dtype=np.uint8).reshape(-1)
    available = bits.size // seq_length
    if n_sequences is None:
        n_sequences = available
    if not 1 <= n_sequences <= available:
        raise ValueError(f"{bits.size} bits give at most {available} sequences of length {seq_length}")
    return bits[: n_sequences * seq_length].reshape(n_sequences, seq_length)


def check_sts_path(path):
    text = str(Path(path).resolve())
    if len(text) >= 200:
        raise ValueError(f"STS can read paths of at most 199 characters ({len(text)} characters): {text}")
    if any(ch.isspace() for ch in text):
        raise ValueError(f"STS cannot read a file whose path contains whitespace: {text}")
    return text


def write_sts_input(path, sequences):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    check_sts_path(path)
    sha256 = hashlib.sha256()
    n_sequences, seq_length = 0, None
    with path.open("wb") as f:
        for bits in sequences:
            bits = np.asarray(bits, dtype=np.uint8).reshape(-1)
            if bits.size == 0 or bits.max() > 1:
                raise ValueError("each sequence must be a non-empty array of 0 and 1")
            if seq_length is None:
                seq_length = bits.size
            elif bits.size != seq_length:
                raise ValueError(f"all sequences must have the same length ({seq_length}), got {bits.size}")
            data = (bits + ord("0")).tobytes()   # 0 -> '0', 1 -> '1'
            f.write(data)
            sha256.update(data)
            n_sequences += 1
    if n_sequences == 0:
        raise ValueError("no sequences were given")
    return {"path": path, "n_sequences": n_sequences, "seq_length": seq_length,
            "n_bits": n_sequences * seq_length, "sha256": sha256.hexdigest()}


def save_sequences(sequences, out_dir, label):
    # Pass the sequences through unchanged, saving each one as its own file <label>_seq<number>.txt in out_dir.
    # out_dir = None saves nothing. Existing files are never overwritten.
    for i, bits in enumerate(sequences, start=1):
        bits = np.asarray(bits, dtype=np.uint8).reshape(-1)
        if out_dir is not None:
            path = Path(out_dir) / f"{label}_seq{i:03d}.txt"
            path.parent.mkdir(parents=True, exist_ok=True)
            with path.open("x", encoding="ascii") as f:
                f.write((bits + ord("0")).tobytes().decode("ascii"))
        yield bits

<!-- ### 3.3 入力ファイルを作る -->

### 3.3 Creating the input file

<!-- `BITS_FILE = None` なら C1 回路の系列を 1 本ずつシミュレーションしながら書き、ファイルを指定した場合はそのビット列を `SEQUENCE_LENGTH` ビットずつに分けて書きます。 -->

If `BITS_FILE = None`, the C1-circuit sequences are simulated and written one at a time; if a file is specified, its bit string is split into chunks of `SEQUENCE_LENGTH` bits and written.

<!-- `SAVE_RESULTS = True` なら、各系列を `RUN_DIR/sequences/` に 1 本 1 ファイルで保存し、STS に渡した入力ファイルの記録を `<data_label>_sts_input.json` に保存します。 -->

If `SAVE_RESULTS = True`, each sequence is saved as its own file in `RUN_DIR/sequences/`, and a record of the input file passed to STS is saved as `<data_label>_sts_input.json`.

In [ ]:
BITS_FILE = None   # e.g., Path("results/toeplitz/2026-11-10_153012/toeplitz_extracted.txt")
N_SEQUENCES = 55
SEQUENCE_LENGTH = 1_000_000
GATE = "H"
READOUT_P1_GIVEN_0 = 0.03
READOUT_P0_GIVEN_1 = 0.03
SIM_SEED = 1

started = time.time()
if BITS_FILE is None:
    data_label = f"c1_{GATE}_sim"
    noise_model = build_noise_model(READOUT_P1_GIVEN_0, READOUT_P0_GIVEN_1)
    sequences = c1_sequences(N_SEQUENCES, SEQUENCE_LENGTH, GATE, noise_model, seed=SIM_SEED)
else:
    data_label = Path(BITS_FILE).stem
    file_bits = load_bits(BITS_FILE)
    n_available = file_bits.size // SEQUENCE_LENGTH
    if n_available < N_SEQUENCES:
        print(f"Note: {file_bits.size:,} bits give only {n_available} sequences ({N_SEQUENCES} requested).")
    sequences = split_sequences(file_bits, SEQUENCE_LENGTH, min(N_SEQUENCES, n_available))
seq_dir = None if RUN_DIR is None else RUN_DIR / "sequences"
if seq_dir is not None and seq_dir.exists():
    raise FileExistsError(f"Destination already exists: {seq_dir}. Run the setup cell (2.) again to save in a new folder.")
# STS reads one file; when saving, each sequence is also saved as its own file in RUN_DIR/sequences
sts_input = write_sts_input(DATA_DIR / f"{data_label}.txt", save_sequences(sequences, seq_dir, data_label))
print(f"{sts_input['path']}: {sts_input['n_sequences']} sequences x {sts_input['seq_length']:,} bits"
      f" ({time.time() - started:.0f} s)")
print("SHA-256:", sts_input["sha256"])
if RUN_DIR is not None:
    # Record of the input file passed to STS (the file itself is in a temporary folder)
    record = {"data": data_label, "sha256": sts_input["sha256"], "n_sequences": sts_input["n_sequences"],
              "seq_length": sts_input["seq_length"], "sequence_files": f"sequences/{data_label}_seq001.txt ...",
              "source": str(BITS_FILE) if BITS_FILE is not None else "simulation",
              "simulation": None if BITS_FILE is not None else {
                  "gate": GATE, "readout_p1_given_0": READOUT_P1_GIVEN_0,
                  "readout_p0_given_1": READOUT_P0_GIVEN_1, "seed": SIM_SEED}}
    with (RUN_DIR / f"{data_label}_sts_input.json").open("x", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, indent=2) + "\n")
    print("Saved:", seq_dir, "and", RUN_DIR / f"{data_label}_sts_input.json")

<!-- ## 4. STS で検定する -->

## 4. Testing with STS

<!-- ### 4.1 STS を実行して報告を読む関数 -->

### 4.1 Functions for running STS and reading the report

<!-- `run_sts` は `STS_DIR` の中で `./assess <n>` を実行し、1. の表の答えを標準入力から渡します。終わったら結果のフォルダを `out_dir` にコピーします（STS は次の実行で同じ場所に上書きするため）。`parse_sts_report` は報告の 188 行を読み取り、`show_sts_summary` と `show_flagged_rows` は検定ごとのまとめと `*` のついた行を表示します。 -->

`run_sts` runs `./assess <n>` inside `STS_DIR` and passes the answers from the table in 1. through standard input. When it finishes, it copies the results folder to `out_dir` (because STS overwrites the same location on the next run). `parse_sts_report` reads the 188 rows of the report, and `show_sts_summary` and `show_flagged_rows` display a per-test summary and the rows marked with `*`.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `run_sts` | STS の `assess` を実行して 15 種類の検定を行い、指定があれば結果のフォルダを保存先にコピーすることを目的とした関数です。 | 結果のフォルダのパス（`out_dir`、または `STS_DIR/experiments/AlgorithmTesting`） | `run_sts(sts_input["path"], 1_000_000, 55, out_dir=RUN_DIR / "sts_results")` |
| `parse_sts_report` | STS の報告 `finalAnalysisReport.txt` の各行を読み取り、Python で扱える辞書に変換することを目的とした関数です。 | `path`, `generator`, `rows`, `min_pass`, `min_pass_excursion` をキーとする辞書 | `parse_sts_report(RUN_DIR / "sts_results")` |
| `variant_labels` | 1 本の系列で複数の P 値を出す検定について、報告の各行が何を表すかの名前（テンプレートや $x$ の値など）を作ることを目的とした関数です。 | 1 本の系列で複数の P 値を出す検定の、各行の意味の文字列のリスト | `variant_labels("RandomExcursions")` |
| `row_label` | 報告の 1 行が何を表すかの名前を求めることを目的とした関数です。 | 報告の 1 行の意味の文字列 | `row_label(report["rows"][0])` |
| `show_sts_summary` | 検定ごとに、`*` のついた行の数と、合格の割合が最も低い行の合格数をまとめて、表で表示することを目的とした関数です。 | 検定ごとの表を表示する。戻り値は `None` | `show_sts_summary({"C1": report})` |
| `show_flagged_rows` | `*` のついた行だけを取り出して表示することを目的とした関数です。 | `*` のついた行を表示する。戻り値はその行のリスト | `show_flagged_rows(report)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `run_sts` | Runs STS's `assess` to perform the 15 tests and, if specified, copies the results folder to a destination. | Path of the results folder (`out_dir`, or `STS_DIR/experiments/AlgorithmTesting`) | `run_sts(sts_input["path"], 1_000_000, 55, out_dir=RUN_DIR / "sts_results")` |
| `parse_sts_report` | Reads each row of the STS report `finalAnalysisReport.txt` and converts it into a dictionary that Python can work with. | Dictionary with the keys `path`, `generator`, `rows`, `min_pass`, and `min_pass_excursion` | `parse_sts_report(RUN_DIR / "sts_results")` |
| `variant_labels` | For tests that give several P-values per sequence, makes the names of what each report row represents (a template, a value of $x$, and so on). | List of strings giving the meaning of each row of a test that gives several P-values per sequence | `variant_labels("RandomExcursions")` |
| `row_label` | Gets the name of what one report row represents. | String giving the meaning of the report row | `row_label(report["rows"][0])` |
| `show_sts_summary` | For each test, summarizes in a table the number of rows marked with `*` and the pass count of the row with the lowest pass proportion. | Displays the per-test table. Returns `None` | `show_sts_summary({"C1": report})` |
| `show_flagged_rows` | Extracts and displays only the rows marked with `*`. | Displays the rows marked with `*`. Returns a list of those rows | `show_flagged_rows(report)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="6"><code>run_sts</code></td><td><code>input_path</code></td><td><code>write_sts_input</code> で作ったファイル</td><td>必須</td></tr>
<tr><td><code>seq_length</code></td><td>1 本の系列のビット数 $n$</td><td>必須</td></tr>
<tr><td><code>n_sequences</code></td><td>系列の本数 $s$</td><td>必須</td></tr>
<tr><td><code>sts_dir</code></td><td>STS のフォルダ。<code>None</code> なら <code>STS_DIR</code></td><td><code>None</code></td></tr>
<tr><td><code>out_dir</code></td><td>結果をコピーするフォルダ。<code>None</code> ならコピーしない</td><td><code>None</code></td></tr>
<tr><td><code>timeout</code></td><td>打ち切るまでの秒数。<code>None</code> なら打ち切らない</td><td><code>None</code></td></tr>
<tr><td><code>parse_sts_report</code></td><td><code>path</code></td><td><code>finalAnalysisReport.txt</code> か、それを含むフォルダ</td><td>必須</td></tr>
<tr><td rowspan="2"><code>variant_labels</code></td><td><code>test</code></td><td>検定の名前（STS の名前）</td><td>必須</td></tr>
<tr><td><code>sts_dir</code></td><td>テンプレートを読む STS のフォルダ。<code>None</code> なら <code>STS_DIR</code></td><td><code>None</code></td></tr>
<tr><td rowspan="2"><code>row_label</code></td><td><code>row</code></td><td><code>parse_sts_report</code> の <code>rows</code> の要素</td><td>必須</td></tr>
<tr><td><code>sts_dir</code></td><td>STS のフォルダ</td><td><code>None</code></td></tr>
<tr><td><code>show_sts_summary</code></td><td><code>reports</code></td><td>列の見出し → <code>parse_sts_report</code> の戻り値 の辞書</td><td>必須</td></tr>
<tr><td rowspan="2"><code>show_flagged_rows</code></td><td><code>report</code></td><td><code>parse_sts_report</code> の戻り値</td><td>必須</td></tr>
<tr><td><code>sts_dir</code></td><td>STS のフォルダ</td><td><code>None</code></td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="6"><code>run_sts</code></td><td><code>input_path</code></td><td>File created by <code>write_sts_input</code></td><td>Required</td></tr>
<tr><td><code>seq_length</code></td><td>Number of bits $n$ per sequence</td><td>Required</td></tr>
<tr><td><code>n_sequences</code></td><td>Number of sequences $s$</td><td>Required</td></tr>
<tr><td><code>sts_dir</code></td><td>STS folder. <code>None</code> means <code>STS_DIR</code></td><td><code>None</code></td></tr>
<tr><td><code>out_dir</code></td><td>Folder to copy the results to. <code>None</code> means do not copy</td><td><code>None</code></td></tr>
<tr><td><code>timeout</code></td><td>Number of seconds before giving up. <code>None</code> means no limit</td><td><code>None</code></td></tr>
<tr><td><code>parse_sts_report</code></td><td><code>path</code></td><td><code>finalAnalysisReport.txt</code> or the folder containing it</td><td>Required</td></tr>
<tr><td rowspan="2"><code>variant_labels</code></td><td><code>test</code></td><td>Name of the test (name in STS)</td><td>Required</td></tr>
<tr><td><code>sts_dir</code></td><td>STS folder from which to read the templates. <code>None</code> means <code>STS_DIR</code></td><td><code>None</code></td></tr>
<tr><td rowspan="2"><code>row_label</code></td><td><code>row</code></td><td>An element of <code>rows</code> returned by <code>parse_sts_report</code></td><td>Required</td></tr>
<tr><td><code>sts_dir</code></td><td>STS folder</td><td><code>None</code></td></tr>
<tr><td><code>show_sts_summary</code></td><td><code>reports</code></td><td>Dictionary of column heading → return value of <code>parse_sts_report</code></td><td>Required</td></tr>
<tr><td rowspan="2"><code>show_flagged_rows</code></td><td><code>report</code></td><td>Return value of <code>parse_sts_report</code></td><td>Required</td></tr>
<tr><td><code>sts_dir</code></td><td>STS folder</td><td><code>None</code></td></tr>
</tbody>
</table>

<br>

<!-- `parse_sts_report` の `rows` の各要素は、`test`（検定の名前）、`index`（その検定の中で何番目の行か、0 から）、`counts`（`C1`〜`C10`）、`uniformity`（$\text{P-value}_T$。`----` なら `None`）、`uniformity_flag`（基準 2 の `*`）、`passed` と `total`（合格数と系列の数。`------` なら `None`）、`proportion_flag`（基準 1 の `*`）をキーとする辞書です。`min_pass` は報告の最後に書かれた基準 1 の最小の合格数と系列の数の組です。 -->

Each element of `rows` returned by `parse_sts_report` is a dictionary with the keys `test` (name of the test), `index` (position of the row within the test, from 0), `counts` (`C1`–`C10`), `uniformity` ($\text{P-value}_T$; `None` if `----`), `uniformity_flag` (`*` for criterion 2), `passed` and `total` (the pass count and the number of sequences; `None` if `------`), and `proportion_flag` (`*` for criterion 1). `min_pass` is the pair of the minimum pass count for criterion 1 and the number of sequences, as written at the end of the report.

In [ ]:
# STS's internal order of tests (the order of the report rows) and the number of P-values per sequence
STS_TESTS = ["Frequency", "BlockFrequency", "CumulativeSums", "Runs", "LongestRun", "Rank", "FFT",
             "NonOverlappingTemplate", "OverlappingTemplate", "Universal", "ApproximateEntropy",
             "RandomExcursions", "RandomExcursionsVariant", "Serial", "LinearComplexity"]
STS_ROWS = {test: 1 for test in STS_TESTS} | {"CumulativeSums": 2, "NonOverlappingTemplate": 148,
                                               "RandomExcursions": 8, "RandomExcursionsVariant": 18, "Serial": 2}


def run_sts(input_path, seq_length, n_sequences, sts_dir=None, out_dir=None, timeout=None):
    sts_dir = Path(STS_DIR if sts_dir is None else sts_dir).resolve()
    input_text = check_sts_path(input_path)
    n_bits_in_file = Path(input_path).stat().st_size   # in ASCII format without separators, 1 byte = 1 bit
    if n_bits_in_file < seq_length * n_sequences:
        raise ValueError(f"the file has {n_bits_in_file} bits, but {n_sequences} x {seq_length} bits are needed")
    # answers from the table in 1.: file / path / all tests / default parameters / number of sequences / ASCII format
    answers = f"0\n{input_text}\n1\n0\n{n_sequences}\n0\n"
    result_dir = sts_dir / "experiments" / "AlgorithmTesting"
    report_path = result_dir / "finalAnalysisReport.txt"
    start = time.time()
    completed = subprocess.run(["./assess", str(seq_length)], input=answers, cwd=sts_dir,
                               capture_output=True, text=True, timeout=timeout)
    finished = "Statistical Testing Complete" in completed.stdout
    if not finished or not report_path.is_file() or report_path.stat().st_mtime < start - 2:
        raise RuntimeError("STS did not finish:\n" + completed.stdout[-3000:] + completed.stderr[-3000:])
    print(f"STS: {n_sequences} sequences x {seq_length:,} bits, {time.time() - start:.0f} s")
    if out_dir is not None:
        out_dir = Path(out_dir)
        shutil.copytree(result_dir, out_dir, dirs_exist_ok=True)
        return out_dir
    return result_dir


ROW_PATTERN = re.compile(
    r"^\s*((?:\d+\s+){10})(----|\d+\.\d+)\s*(\*?)\s+(?:(\d+)/(\d+)\s*(\*?)|------)\s+(\w+)\s*$")
MIN_PASS_PATTERN = re.compile(r"random excursion \(variant\) test is approximately = (\d+) for a\s+sample size = (\d+)")
MIN_PASS_EXC_PATTERN = re.compile(r"random excursion \(variant\) test\s+is approximately = (\d+) for a sample size = (\d+)")


def parse_sts_report(path):
    path = Path(path)
    if path.is_dir():
        path = path / "finalAnalysisReport.txt"
    text = path.read_text(encoding="ascii", errors="replace")
    rows, seen = [], {}
    for line in text.splitlines():
        m = ROW_PATTERN.match(line)
        if not m:
            continue
        test = m.group(7)
        index = seen.get(test, 0)
        seen[test] = index + 1
        rows.append({
            "test": test, "index": index,
            "counts": [int(c) for c in m.group(1).split()],
            "uniformity": None if m.group(2) == "----" else float(m.group(2)),
            "uniformity_flag": m.group(3) == "*",
            "passed": None if m.group(4) is None else int(m.group(4)),
            "total": None if m.group(5) is None else int(m.group(5)),
            "proportion_flag": m.group(6) == "*",
        })
    generator = re.search(r"generator is <(.*)>", text)
    min_pass = MIN_PASS_PATTERN.search(text)
    min_pass_exc = MIN_PASS_EXC_PATTERN.search(text)
    return {
        "path": path,
        "generator": generator.group(1) if generator else None,
        "rows": rows,
        "min_pass": tuple(map(int, min_pass.groups())) if min_pass else None,             # (minimum pass count, s)
        "min_pass_excursion": tuple(map(int, min_pass_exc.groups())) if min_pass_exc else None,
    }


def variant_labels(test, sts_dir=None):
    if test == "CumulativeSums":
        return ["forward", "backward"]
    if test == "Serial":
        return ["P-value 1", "P-value 2"]
    if test == "RandomExcursions":
        return [f"x = {x:+d}" for x in (-4, -3, -2, -1, 1, 2, 3, 4)]
    if test == "RandomExcursionsVariant":
        return [f"x = {x:+d}" for x in [*range(-9, 0), *range(1, 10)]]
    if test == "NonOverlappingTemplate":
        template_file = Path(STS_DIR if sts_dir is None else sts_dir) / "templates" / "template9"
        if template_file.is_file():
            lines = [line for line in template_file.read_text().splitlines() if line.strip()]
            return ["B = " + "".join(line.split()) for line in lines]
        return [f"template {i + 1}" for i in range(STS_ROWS[test])]
    return [""]


def row_label(row, sts_dir=None):
    labels = variant_labels(row["test"], sts_dir)
    return labels[row["index"]] if row["index"] < len(labels) else f"#{row['index'] + 1}"


def show_sts_summary(reports):
    labels = list(reports)
    lines = ["| test | rows | " + " | ".join(f"{label}: `*` rows (min passed / total)" for label in labels) + " |",
             "|---|---|" + "---|" * len(labels)]
    for test in STS_TESTS:
        cells, n_rows = [], None
        for label in labels:
            rows = [r for r in reports[label]["rows"] if r["test"] == test]
            n_rows = len(rows)
            flagged = sum(r["proportion_flag"] or r["uniformity_flag"] for r in rows)
            counted = [r for r in rows if r["total"]]
            lowest = min(counted, key=lambda r: r["passed"] / r["total"]) if counted else None
            text = f"{flagged}" + (f" ({lowest['passed']}/{lowest['total']})" if lowest else " (------)")
            cells.append(f"**{text}**" if flagged else text)
        lines.append(f"| {test} | {n_rows} | " + " | ".join(cells) + " |")
    totals = [sum(r["proportion_flag"] or r["uniformity_flag"] for r in reports[label]["rows"]) for label in labels]
    n_total = len(reports[labels[0]]["rows"])
    lines.append(f"| **total** | {n_total} | " + " | ".join(f"**{t}**" for t in totals) + " |")
    display(Markdown("\n".join(lines)))


def show_flagged_rows(report, sts_dir=None):
    flagged = [r for r in report["rows"] if r["proportion_flag"] or r["uniformity_flag"]]
    if not flagged:
        display(Markdown("No rows are marked with `*`."))
        return flagged
    lines = ["| test | row | C1 ... C10 | P-value_T | proportion |", "|---|---|---|---|---|"]
    for r in flagged[:40]:
        uniformity = "----" if r["uniformity"] is None else f"{r['uniformity']:.6f}" + (" *" if r["uniformity_flag"] else "")
        proportion = "------" if r["total"] is None else f"{r['passed']}/{r['total']}" + (" *" if r["proportion_flag"] else "")
        lines.append(f"| {r['test']} | {row_label(r, sts_dir)} | {' '.join(map(str, r['counts']))} | {uniformity} | {proportion} |")
    if len(flagged) > 40:
        lines.append(f"| ... | {len(flagged) - 40} more rows | | | |")
    display(Markdown("\n".join(lines)))
    return flagged

<!-- ### 4.2 検定を実行する -->

### 4.2 Running the tests

<!-- 3.3 のファイルを STS で検定します。`SAVE_RESULTS = True` なら、STS の結果のフォルダを `RUN_DIR/sts_results` にコピーします。検定ごとのまとめの各セルは「`*` のついた行の数（合格数が最も少ない行の合格数 / 系列の数）」で、その後に `*` のついた行を表示します。読み方は手法の但し書きを参照してください。 -->

We test the file from 3.3 with STS. If `SAVE_RESULTS = True`, the STS results folder is copied to `RUN_DIR/sts_results`. Each cell of the per-test summary shows "the number of rows marked with `*` (pass count of the row with the fewest passes / number of sequences)", and the rows marked with `*` are displayed afterwards. For how to read them, see the caveats in Methods.

In [ ]:
if STS_AVAILABLE:
    sts_out = None if RUN_DIR is None else RUN_DIR / "sts_results"
    if sts_out is not None and sts_out.exists():
        raise FileExistsError(f"Destination already exists: {sts_out}. Run the setup cell (2.) again to save in a new folder.")
    result_dir = run_sts(sts_input["path"], sts_input["seq_length"], sts_input["n_sequences"], out_dir=sts_out)
    report = parse_sts_report(result_dir)
    print("Rows read:", len(report["rows"]))
    print("Minimum pass count for criterion 1 (passes, s):", report["min_pass"],
          "/ Random Excursions:", report["min_pass_excursion"])
    show_sts_summary({data_label: report})
    flagged_rows = show_flagged_rows(report)
else:
    print("STS is not available, so this cell is skipped.")

<!-- ## 5. 結果を保存する -->

## 5. Saving the results

<!-- `SAVE_RESULTS = True` なら、報告の 188 行を `RUN_DIR/<data_label>_sts.csv` に保存します。同じ `RUN_DIR` には、STS の結果のフォルダのコピー（`sts_results/`）と、検定した系列（`sequences/`）があります。 -->

If `SAVE_RESULTS = True`, the 188 rows of the report are saved to `RUN_DIR/<data_label>_sts.csv`. The same `RUN_DIR` holds a copy of the STS results folder (`sts_results/`) and the tested sequences (`sequences/`).

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `save_sts_csv` | 報告の全行を、後から解析できるよう CSV ファイルに保存することを目的とした関数です。 | 保存したファイルのパス | `save_sts_csv(RUN_DIR / "c1_H_sim_sts.csv", {"C1": report})` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `save_sts_csv` | Saves all rows of the report to a CSV file for later analysis. | Path of the saved file | `save_sts_csv(RUN_DIR / "c1_H_sim_sts.csv", {"C1": report})` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="3"><code>save_sts_csv</code></td><td><code>path</code></td><td>保存先。既に存在する場合は上書き</td><td>必須</td></tr>
<tr><td><code>reports</code></td><td>見出し → <code>parse_sts_report</code> の戻り値 の辞書</td><td>必須</td></tr>
<tr><td><code>sts_dir</code></td><td>テンプレートを読む STS のフォルダ。<code>None</code> なら <code>STS_DIR</code></td><td><code>None</code></td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="3"><code>save_sts_csv</code></td><td><code>path</code></td><td>Destination. Overwritten if it already exists</td><td>Required</td></tr>
<tr><td><code>reports</code></td><td>Dictionary of heading → return value of <code>parse_sts_report</code></td><td>Required</td></tr>
<tr><td><code>sts_dir</code></td><td>STS folder from which to read the templates. <code>None</code> means <code>STS_DIR</code></td><td><code>None</code></td></tr>
</tbody>
</table>

In [ ]:
def save_sts_csv(path, reports, sts_dir=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["data", "test", "row", "variant"] + [f"C{i}" for i in range(1, 11)]
                        + ["uniformity_p", "uniformity_flag", "passed", "total", "proportion_flag", "generator"])
        for label, report in reports.items():
            for r in report["rows"]:
                writer.writerow([label, r["test"], r["index"] + 1, row_label(r, sts_dir)] + r["counts"]
                                + ["" if r["uniformity"] is None else r["uniformity"], int(r["uniformity_flag"]),
                                   "" if r["passed"] is None else r["passed"], "" if r["total"] is None else r["total"],
                                   int(r["proportion_flag"]), report["generator"]])
    return path


if not STS_AVAILABLE:
    print("There are no STS results, so nothing is saved.")
elif RUN_DIR is None:
    print("SAVE_RESULTS = False, so the report is not saved.")
else:
    csv_path = RUN_DIR / f"{data_label}_sts.csv"
    if csv_path.exists():
        raise FileExistsError(f"Destination already exists: {csv_path}. Run the setup cell (2.) again to save in a new folder.")
    print("Saved to:", save_sts_csv(csv_path, {data_label: report}))